# Práctica guiada: AWS SageMaker

**AWS SageMaker** es una plataforma para crear, entrenar e implementar modelos de IA y machine learning.

En esta actividad vamos a utilizar SageMaker para entrenar un modelo de XGBoost con datos de costo de vivienda en México. Se utiliza el proyecto [Housing-in-Mexico](https://github.com/IgorTraspadini/Housing-in-Mexico).

## 1. Abrir notebook

--> Ve a "AWS Console"

--> busca el servicio "Amazon SageMaker AI"

--> en el panel izquierdo "Notebooks"

--> presiona el botón "Crea un notebook"

## 2. Carga los datos del proyecto

In [2]:
import pandas as pd

repo_url = "https://raw.githubusercontent.com/IgorTraspadini/Housing-in-Mexico/refs/heads/main"
csv_url = f"{repo_url}/mexico-real-state-clean.csv"
df = pd.read_csv(csv_url)

df.head()

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150,67965.56
1,house,Nuevo León,25.688436,-100.198807,186,63223.78
2,apartment,Guerrero,16.767704,-99.764383,82,84298.37
3,apartment,Guerrero,16.829782,-99.911012,150,94308.80
4,house,Yucatán,21.052583,-89.538639,205,105191.37


## 3. Divide los datos

Divide el dataset en 3 subsets, uno de entrenamiento, uno de validación y uno de pruebas.

Después guarda estos datasets como archivos CSV en S3.

In [3]:
import os
import sys
import io
import time
import uuid
import tarfile
import subprocess
from io import StringIO

import boto3
import numpy as np
import sagemaker
import xgboost as xgb
from sagemaker.inputs import TrainingInput
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score

print(f"Versión de Python: {sys.version}")
print(f"Versión de Boto3: {boto3.__version__}")
print(f"Versión de XGBoost: {xgb.__version__}")
!aws --version

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Versión de Python: 3.10.20 | packaged by conda-forge | (main, Jun 11 2026, 03:31:56) [GCC 14.3.0]
Versión de Boto3: 1.43.124
Versión de XGBoost: 1.7.1
aws-cli/1.46.2 Python/3.10.20 Linux/6.12.113-138.209.amzn2023.x86_64 botocore/1.43.62


In [4]:
role = sagemaker.get_execution_role()
region_name = sagemaker.Session().boto_region_name

# Identificador aleatorio para que el nombre del bucket sea único sin
# exponer el account_id (este notebook va a vivir en un repo público).
run_id = uuid.uuid4().hex[:8]

print(f"Región: {region_name}")
print(f"Run ID: {run_id}")

Región: us-east-1
Run ID: 5eba9add


In [5]:
# Nombre base del proyecto y bucket de S3.
# run_id hace que el nombre del bucket sea único para cada alumno
# (los nombres de bucket son únicos en todo S3, no solo en tu cuenta).
base_name = "housing-price-mexico"
s3_bucket = f"{base_name}-bucket-{run_id}"

s3_resource = boto3.resource("s3")
s3_bucket_resource = s3_resource.Bucket(s3_bucket)
s3_resource.create_bucket(Bucket=s3_bucket)

sm_client = boto3.client("sagemaker")

print(f"Bucket: {s3_bucket}")

Bucket: housing-price-mexico-bucket-5eba9add


In [6]:
# Nombres de los recursos
train_job_name = f"train-{base_name}"

# Subdirectorios locales dentro del notebook
data_dir = os.path.join(os.getcwd(), f"data/{base_name}")
train_dir = os.path.join(data_dir, "train")
val_dir = os.path.join(data_dir, "validate")
test_dir = os.path.join(data_dir, "test")

# Directorio local donde se van a extraer los artefactos del modelo
container_artifacts_dir = os.path.join(os.getcwd(), f"container-artifacts/{base_name}")

# Prefijos en S3
train_dir_s3_prefix = f"{base_name}/data/train"
val_dir_s3_prefix = f"{base_name}/data/validate"
test_dir_s3_prefix = f"{base_name}/data/test"

# Ubicación en S3 donde se van a guardar los checkpoints y el modelo entrenado
model_checkpoint_s3_path = f"s3://{s3_bucket}/{base_name}/checkpoint/"
model_output_s3_path = f"s3://{s3_bucket}/{base_name}/output/"

# Nombres del archivo tar del modelo y del archivo extraído. Dependen del
# algoritmo con el que se entrenó: el algoritmo built-in de XGBoost de
# SageMaker siempre los nombra así.
model_tar_file_name = "model.tar.gz"
extracted_model_file_name = "xgboost-model"

In [7]:
!pip install -q scikit-learn xgboost==1.7.1


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [8]:
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)
os.makedirs(container_artifacts_dir, exist_ok=True)

In [9]:
# ─── 1. Separar features ───
cat_cols = ["property_type", "state"]
num_cols = ["lat", "lon", "area_m2"]

# ─── 2. One-hot encode sobre el dataset COMPLETO (antes del split) ───
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat = ohe.fit_transform(df[cat_cols])

# Combinar: numéricas + one-hot
X = np.hstack([df[num_cols].to_numpy(), X_cat])
y = df["price_usd"]

feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
print("Features:", feature_names)

# ─── 3. Split 80/10/10 ───
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

Features: ['lat', 'lon', 'area_m2', 'property_type_apartment', 'property_type_house', 'state_Aguascalientes', 'state_Baja California', 'state_Baja California Sur', 'state_Campeche', 'state_Chiapas', 'state_Chihuahua', 'state_Colima', 'state_Distrito Federal', 'state_Durango', 'state_Estado de México', 'state_Guanajuato', 'state_Guerrero', 'state_Hidalgo', 'state_Jalisco', 'state_Morelos', 'state_Nayarit', 'state_Nuevo León', 'state_Oaxaca', 'state_Puebla', 'state_Querétaro', 'state_Quintana Roo', 'state_San Luis Potosí', 'state_Sinaloa', 'state_Sonora', 'state_Tabasco', 'state_Tamaulipas', 'state_Tlaxcala', 'state_Veracruz de Ignacio de la Llave', 'state_Yucatán', 'state_Zacatecas']
X_train shape: (1388, 35)
X_val shape: (174, 35)
X_test shape: (174, 35)


In [10]:
# ─── 4. Guardar CSVs ───
np.savetxt(os.path.join(train_dir, "train.csv"),
           np.concatenate((y_train.to_numpy().reshape(-1, 1), X_train), axis=1), delimiter=",")
np.savetxt(os.path.join(train_dir, "train_x.csv"), X_train)
np.savetxt(os.path.join(train_dir, "train_y.csv"), y_train.to_numpy())

np.savetxt(os.path.join(val_dir, "validate.csv"),
           np.concatenate((y_val.to_numpy().reshape(-1, 1), X_val), axis=1), delimiter=",")
np.savetxt(os.path.join(val_dir, "validate_x.csv"), X_val)
np.savetxt(os.path.join(val_dir, "validate_y.csv"), y_val.to_numpy())

np.savetxt(os.path.join(test_dir, "test.csv"),
           np.concatenate((y_test.to_numpy().reshape(-1, 1), X_test), axis=1), delimiter=",")
np.savetxt(os.path.join(test_dir, "test_x.csv"), X_test)
np.savetxt(os.path.join(test_dir, "test_y.csv"), y_test.to_numpy())

print("CSVs guardados")

CSVs guardados


In [11]:
# Subir los datos a S3
train_s3_path = sagemaker.Session().upload_data(train_dir, bucket=s3_bucket, key_prefix=train_dir_s3_prefix)
val_s3_path = sagemaker.Session().upload_data(val_dir, bucket=s3_bucket, key_prefix=val_dir_s3_prefix)
test_s3_path = sagemaker.Session().upload_data(test_dir, bucket=s3_bucket, key_prefix=test_dir_s3_prefix)

print(train_s3_path)
print(val_s3_path)
print(test_s3_path)

s3://housing-price-mexico-bucket-5eba9add/housing-price-mexico/data/train
s3://housing-price-mexico-bucket-5eba9add/housing-price-mexico/data/validate
s3://housing-price-mexico-bucket-5eba9add/housing-price-mexico/data/test


Ve a ver S3 en la consola. ¿Están ahí los archivos?

Ver los archivos locales

```bash
data
└── housing-price-mexico
    ├── test
    │   ├── test.csv
    │   ├── test_x.csv
    │   └── test_y.csv
    ├── train
    │   ├── train.csv
    │   ├── train_x.csv
    │   └── train_y.csv
    └── validate
        ├── validate.csv
        ├── validate_x.csv
        └── validate_y.csv
```

## 4. Entrenar un modelo

Pasos para obtener un modelo predictivo

In [12]:
import logging

# ─── Inputs ───
train_input = TrainingInput(f"{train_s3_path}/train.csv", content_type="text/csv")
val_input = TrainingInput(f"{val_s3_path}/validate.csv", content_type="text/csv")
inputs = {"train": train_input, "validation": val_input}

# ─── Hyperparameters ───
hyperparameters = {
    "objective": "reg:squarederror",
    "max_depth": "4",              # 7 es profundo para ~1700 filas
    "eta": "0.05",                 # subo un poco para no necesitar 1864 rondas
    "num_round": "600",
    "subsample": "0.8",            # regulariza fila a fila
    "colsample_bytree": "0.7",
    "min_child_weight": "3",       # más robusto con categorías de state poco pobladas
    "alpha": "1.0",
    "lambda": "2.0",               # L2
    "early_stopping_rounds": "20", # corta solo cuando deja de mejorar en validation
    "verbosity": "0",
    "seed": "42",
}

# ─── Instance ───
train_instance_count = 1
train_instance_type = "ml.m5.xlarge"
max_run_time_in_seconds = 3600

# ─── Container ───
container_image_uri = "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1"

# ─── Paths ───
model_checkpoint_local_dir = "/opt/ml/checkpoints/"
model_local_dir = "/opt/ml/model"

# ─── Borrar checkpoints de corridas anteriores ───
for checkpoint_file in s3_bucket_resource.objects.filter(Prefix=f"{base_name}/checkpoint/"):
    print(f"Eliminando {checkpoint_file.key} ...")
    s3_resource.Object(s3_bucket, checkpoint_file.key).delete()

# ─── Silenciar logs ───
logging.getLogger("sagemaker").setLevel(logging.CRITICAL)
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

# ─── Estimator ───
estimator = sagemaker.estimator.Estimator(
    image_uri=container_image_uri,
    checkpoint_local_path=model_checkpoint_local_dir,
    checkpoint_s3_uri=model_checkpoint_s3_path,
    model_dir=model_local_dir,
    output_path=model_output_s3_path,
    instance_type=train_instance_type,
    instance_count=train_instance_count,
    max_run=max_run_time_in_seconds,
    hyperparameters=hyperparameters,
    role=role,
    base_job_name=train_job_name,
    container_log_level=logging.CRITICAL,
    script_mode=False,
    debugger_hook_config=False,
    disable_profiler=True,
)

# ─── Train ───
estimator.fit(inputs, wait=True, logs="None")


2026-11-18 20:24:21 Starting - Starting the training job..
2026-11-18 20:24:36 Starting - Preparing the instances for training...
2026-11-18 20:25:00 Downloading - Downloading input data....
2026-11-18 20:25:25 Downloading - Downloading the training image........
2026-11-18 20:26:10 Training - Training image download completed. Training in progress....
2026-11-18 20:26:26 Uploading - Uploading generated training model.
2026-11-18 20:26:39 Completed - Training job completed


## 5. Validación

en AWS Console ve a Amazon SageMaker AI --> Model training & customization --> Training & tuning jobs

In [13]:
# Descargar y extraer el modelo entrenado
model_tar_file_s3_path_suffix = f"{base_name}/output/{estimator.latest_training_job.name}/output/{model_tar_file_name}"
model_tar_file_local_path = os.path.join(container_artifacts_dir, model_tar_file_name)
extracted_model_file_local_path = os.path.join(container_artifacts_dir, extracted_model_file_name)

if os.path.exists(model_tar_file_local_path):
    os.remove(model_tar_file_local_path)
if os.path.exists(extracted_model_file_local_path):
    os.remove(extracted_model_file_local_path)

s3_bucket_resource.download_file(model_tar_file_s3_path_suffix, model_tar_file_local_path)

with tarfile.open(model_tar_file_local_path, "r:gz") as tar:
    tar.extractall(path=container_artifacts_dir)

model = xgb.Booster()
model.load_model(extracted_model_file_local_path)

In [14]:
# ─── Helper sencillo: predecir con datos en texto plano ───
# Usamos el mismo `ohe` que ya entrenamos en la sección 3, en vez de
# mantener una lista de categorías aparte a mano.
def predecir_precio(property_type, state, lat, lon, area_m2):
    """Recibe los datos de una propiedad en texto plano y regresa el precio estimado en USD."""
    if property_type not in ohe.categories_[0]:
        raise ValueError(f"property_type debe ser uno de {list(ohe.categories_[0])}")
    if state not in ohe.categories_[1]:
        raise ValueError(f"state debe ser uno de {list(ohe.categories_[1])}")

    cat_encoded = ohe.transform(pd.DataFrame([[property_type, state]], columns=cat_cols))[0]
    fila = [lat, lon, area_m2] + list(cat_encoded)

    pred_df = pd.DataFrame([fila])
    return float(model.predict(xgb.DMatrix(pred_df.values))[0])

In [15]:
# Métricas que calculó el propio contenedor de entrenamiento (train / validation)
job_desc = sm_client.describe_training_job(TrainingJobName=estimator.latest_training_job.name)
for m in job_desc["FinalMetricDataList"]:
    print(f"{m['MetricName']}: {m['Value']:.4f}")

# R² sobre esos mismos datos de entrenamiento y validación
y_pred_train = model.predict(xgb.DMatrix(X_train))
y_pred_val = model.predict(xgb.DMatrix(X_val))
print(f"R² en train: {r2_score(y_train, y_pred_train):.2f}")
print(f"R² en validation: {r2_score(y_val, y_pred_val):.2f}")

# RMSE y R² sobre test — el único conjunto que el modelo nunca vio
y_pred_test = model.predict(xgb.DMatrix(X_test))
rmse = mean_squared_error(y_test, y_pred_test) ** 0.5
r2_test = r2_score(y_test, y_pred_test)
print(f"RMSE en test: ${rmse:,.2f}")
print(f"R² en test: {r2_test:.2f}")

train:rmse: 23790.0469
validation:rmse: 46909.2969
R² en train: 0.86
R² en validation: 0.52
RMSE en test: $45,547.39
R² en test: 0.55


R² indica qué tanto explica el modelo la variación de precios (0 a 1, más cerca de 1 es mejor). El de train/validation puede verse artificialmente bien porque el modelo ya los usó para ajustarse; el de test es el único honesto, porque son datos que el modelo nunca vio.

Ejemplo de uso del modelo

In [16]:
precio = predecir_precio(property_type="house", state="Jalisco", lat=20.67, lon=-103.35, area_m2=120)
print(f"Precio estimado: ${precio:,.2f} USD")

Precio estimado: $71,541.23 USD


Ejemplo con un dato real del test

In [17]:
y_pred_real = model.predict(xgb.DMatrix(X_test[:1]))[0]
print(f"Valor predicho: ${y_pred_real:,.2f}")
print(f"Valor real: ${y_test.values[0]:,.2f}")

Valor predicho: $117,279.16
Valor real: $52,574.87


## 6. Despliegue

curl POST → EC2 (FastAPI, puerto 8080) → xgboost-model → JSON response

In [18]:
# App code
app_code = """from fastapi import FastAPI
from pydantic import BaseModel
import xgboost as xgb
import pandas as pd
from io import StringIO

app = FastAPI()

model = xgb.Booster()
model.load_model("/opt/model/xgboost-model")


class PredictRequest(BaseModel):
    pred_x_csv: str


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(req: PredictRequest):
    pred_x_df = pd.read_csv(StringIO(req.pred_x_csv), sep=",", header=None)
    preds = model.predict(xgb.DMatrix(pred_x_df.values))
    # cast numpy.float32 -> float: FastAPI/Pydantic no pueden serializar escalares de numpy
    return {"Predicted value": float(preds[0])}
"""

# Subir app.py a S3
s3_bucket_resource.upload_fileobj(io.BytesIO(app_code.encode()), f"{base_name}/app.py")

In [19]:
user_data_template = '''#!/bin/bash
exec > /var/log/user_data.log 2>&1
export DEBIAN_FRONTEND=noninteractive

echo "=== START $(date) ==="

echo "=== APT ==="
sudo apt-get update -y || { echo "APT UPDATE FAILED"; exit 1; }
sudo apt-get install -y python3-pip python3-venv || { echo "APT INSTALL FAILED"; exit 1; }

echo "=== PIP ==="
sudo rm -rf /opt/venv
sudo python3 -m venv /opt/venv
sudo /opt/venv/bin/pip install awscli fastapi uvicorn numpy scipy pandas || { echo "PIP BASE FAILED"; exit 1; }

# IMPORTANTE: misma version de xgboost que el contenedor de entrenamiento
# (sagemaker-xgboost:1.7-1 -> xgboost 1.7.1), para que el modelo serializado
# se cargue y prediga igual que en el notebook.
sudo /opt/venv/bin/pip install xgboost==1.7.1 --no-deps || { echo "PIP XGB FAILED"; exit 1; }

echo "=== S3 DOWNLOAD ==="
sudo mkdir -p /opt/model
LATEST_TAR=$(/opt/venv/bin/aws s3 ls s3://__S3_BUCKET__/__BASE_NAME__/output/ --recursive | grep model.tar.gz | sort | tail -1 | awk '{print $4}')
echo "Latest tar: $LATEST_TAR"
if [ -z "$LATEST_TAR" ]; then
  echo "NO MODEL TAR FOUND IN S3"
  exit 1
fi

/opt/venv/bin/aws s3 cp "s3://__S3_BUCKET__/$LATEST_TAR" /tmp/model.tar.gz || { echo "S3 CP MODEL FAILED"; exit 1; }
/opt/venv/bin/aws s3 cp s3://__S3_BUCKET__/__BASE_NAME__/app.py /opt/app.py || { echo "APP CP FAILED"; exit 1; }
sudo tar xzf /tmp/model.tar.gz -C /opt/model/ || { echo "TAR EXTRACT FAILED"; exit 1; }

echo "=== MODEL FILES ==="
ls -la /opt/model/

echo "=== SYSTEMD SERVICE ==="
sudo tee /etc/systemd/system/housing-price.service > /dev/null <<'UNIT'
[Unit]
Description=Housing price FastAPI service
After=network.target

[Service]
Type=simple
WorkingDirectory=/opt
ExecStart=/opt/venv/bin/uvicorn app:app --host 0.0.0.0 --port 8080
Restart=always
RestartSec=3
User=root

[Install]
WantedBy=multi-user.target
UNIT

sudo systemctl daemon-reload
sudo systemctl enable --now housing-price.service

echo "=== DONE $(date) ==="
'''

user_data = user_data_template.replace("__S3_BUCKET__", s3_bucket).replace("__BASE_NAME__", base_name)

In [20]:
ec2 = boto3.client("ec2", region_name=region_name)

existing_instances = ec2.describe_instances(
    Filters=[
        {"Name": "tag:Name", "Values": [f"{base_name}-server"]},
        {"Name": "instance-state-name", "Values": ["pending", "running", "stopping", "stopped"]},
    ]
)
for r in existing_instances["Reservations"]:
    for i in r["Instances"]:
        ec2.terminate_instances(InstanceIds=[i["InstanceId"]])
        print(f"Terminando EC2: {i['InstanceId']}")
        while True:
            state = ec2.describe_instances(InstanceIds=[i["InstanceId"]])["Reservations"][0]["Instances"][0]["State"]["Name"]
            if state == "terminated":
                break
            time.sleep(5)
        print(f"EC2 {i['InstanceId']} destruida")

existing_sg = ec2.describe_security_groups(
    Filters=[{"Name": "group-name", "Values": [f"{base_name}-sg"]}]
)["SecurityGroups"]
if existing_sg:
    ec2.delete_security_group(GroupId=existing_sg[0]["GroupId"])
    print(f"SG {existing_sg[0]['GroupId']} borrado")

# ─── DEPLOY ───
sg = ec2.create_security_group(
    GroupName=f"{base_name}-sg",
    Description="FastAPI app",
)
ec2.authorize_security_group_ingress(
    GroupId=sg["GroupId"],
    IpPermissions=[
        {
            "IpProtocol": "tcp",
            "FromPort": 8080,
            "ToPort": 8080,
            "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "Demo de clase"}],
        },
        {
            "IpProtocol": "tcp",
            "FromPort": 22,
            "ToPort": 22,
            "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "SSH"}],
        },
    ],
)

key_name = f"{base_name}-key"
key_path = f"./{key_name}.pem"
if not os.path.exists(key_path):
    try:
        key = ec2.create_key_pair(KeyName=key_name)
        with open(key_path, "w") as f:
            f.write(key["KeyMaterial"])
        os.chmod(key_path, 0o400)
        print(f"Key pair creado: {os.path.abspath(key_path)}")
    except ec2.exceptions.InvalidKeyPair.Duplicate:
        print(f"Key pair '{key_name}' ya existe en AWS pero no tienes el .pem local.")
        print("Bórralo en la consola de IAM/EC2 y re-correr, o cópialo manualmente.")
else:
    print(f"Key pair local ya existe: {os.path.abspath(key_path)}")

response = ec2.run_instances(
    ImageId="ami-0b6c6ebed2801a5cb",
    InstanceType="t3.micro",
    MinCount=1,
    MaxCount=1,
    SecurityGroupIds=[sg["GroupId"]],
    UserData=user_data,
    KeyName=key_name,
    IamInstanceProfile={"Name": "LabInstanceProfile"},
    TagSpecifications=[{
        "ResourceType": "instance",
        "Tags": [{"Key": "Name", "Value": f"{base_name}-server"}],
    }],
)

instance_id = response["Instances"][0]["InstanceId"]
print(f"EC2 lanzado: {instance_id}")

Key pair creado: /home/ec2-user/SageMaker/housing-price-mexico-key.pem
EC2 lanzado: i-0c3c6262e4c950772


In [21]:
import time; time.sleep(90)

# Esperar running
while ec2.describe_instances(InstanceIds=[instance_id])["Reservations"][0]["Instances"][0]["State"]["Name"] != "running":
    time.sleep(5)
print("EC2 running")

# IP pública
public_ip = ec2.describe_instances(InstanceIds=[instance_id])["Reservations"][0]["Instances"][0]["PublicIpAddress"]
print(f"IP pública: {public_ip}")
print(f"Endpoint: http://{public_ip}:8080/predict")

ssh_base = ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ConnectTimeout=5", "-i", key_path, f"ubuntu@{public_ip}"]

# Esperar a que SSH responda antes de intentar cloud-init status
while subprocess.run(ssh_base + ["true"], capture_output=True).returncode != 0:
    time.sleep(5)
print("SSH listo")

# Bloquea hasta que termine el user_data (éxito o error)
subprocess.run(ssh_base + ["cloud-init", "status", "--wait"], check=True)
print("user_data terminó")

import requests

while True:
    try:
        if requests.get(f"http://{public_ip}:8080/health", timeout=2).status_code == 200:
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)
print("API lista")

EC2 running
IP pública: 52.91.8.153
Endpoint: http://52.91.8.153:8080/predict
SSH listo
................................................................................................................................................................................................status: done
user_data terminó
API lista


In [22]:
# ─── TEST ───
# Misma codificación que usa predecir_precio, con el mismo `ohe`: lat 20.67,
# lon -103.35, area_m2 120, property_type "house", state "Jalisco".
cat_encoded = ohe.transform(pd.DataFrame([["house", "Jalisco"]], columns=cat_cols))[0]
fila = [20.67, -103.35, 120] + list(cat_encoded)
pred_x_csv = ",".join(str(v) for v in fila)

resp = requests.post(f"http://{public_ip}:8080/predict", json={"pred_x_csv": pred_x_csv})
print(f"Respuesta: {resp.json()}")

Respuesta: {'Predicted value': 71541.2265625}


También lo puedes probar directo con `curl`:

In [23]:
body = f'{{"pred_x_csv": "{pred_x_csv}"}}'
!curl -X POST http://{public_ip}:8080/predict -H "Content-Type: application/json" -d '{body}'

{"Predicted value":71541.2265625}